# Custom LLM Guardrail in LangChain — Chunk by Chunk

This notebook demonstrates a **custom output-safety guardrail** inspired by the screenshot.

**Important distinction:** the screenshot shows an `after_agent` hook, which checks the agent's final answer *after* the agent has run. A guardrail that checks a user's request *before* the agent runs should use `before_agent` instead.

We'll use a separate LLM as a safety judge and harmless dummy tools. No emails are sent and no real-world actions are performed.

## Chunk 1 — Install dependencies

Run this cell once in your notebook environment. If you install packages while the kernel is running, restart the kernel if imports still fail.

In [ ]:
%pip install -U langchain langchain-openai langgraph

## Chunk 2 — Imports and API key

Set `OPENAI_API_KEY` in your environment or `.env` file before running this cell. Do not paste your key into a notebook that you plan to share.

In [ ]:
import os
from getpass import getpass

from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage
from langchain.tools import tool
from langgraph.runtime import Runtime
from typing import Any

# If the key isn't already in the environment, ask for it securely.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## Chunk 3 — Create harmless dummy tools

These tools only return pretend results. They do not send emails, access accounts, or change external systems.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Return pretend weather information for a city."""
    return f"Demo weather for {city}: sunny, 24°C (simulated result)."


@tool
def draft_email(to: str, subject: str, body: str) -> str:
    """Create a pretend email draft. This tool does not send email."""
    return (
        "DEMO ONLY — email was NOT sent. "
        f"Draft created for {to}; subject={subject!r}; body={body!r}"
    )


tools = [get_weather, draft_email]
print("Registered dummy tools:", [t.name for t in tools])

## Chunk 4 — Implement the custom safety-judge middleware

The middleware:
1. Reads the final assistant message.
2. Asks a separate model to classify it as `SAFE` or `UNSAFE`.
3. If unsafe, appends a safe replacement message to the conversation state.

The classifier prompt is only a demo. Production systems need stronger policy definitions, testing, and careful handling of false positives/negatives.

In [ ]:
from langchain_openai import ChatOpenAI

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Check the agent's final response with a separate LLM safety judge."""

    def __init__(self):
        super().__init__()
        self.safety_model = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0,
        )

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime,
    ) -> dict[str, Any] | None:
        messages = state.get("messages", [])
        if not messages:
            return None

        last_message = messages[-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Avoid judging an empty response.
        response_text = last_message.content
        if not isinstance(response_text, str) or not response_text.strip():
            return None

        safety_prompt = f'''
You are a safety classifier for an AI assistant's final response.
Evaluate the response for clearly harmful instructions, abuse, or dangerous wrongdoing.
Return exactly one word: SAFE or UNSAFE.

Response to evaluate:
<response>
{response_text}
</response>
'''
        verdict = self.safety_model.invoke(safety_prompt).content
        verdict = str(verdict).strip().upper()

        print(f"[Safety guardrail] Judge verdict: {verdict}")

        if verdict.startswith("UNSAFE"):
            # AgentState's messages use an append-style reducer.
            # Appending this replacement makes it the last assistant message.
            return {
                "messages": [
                    AIMessage(
                        content=(
                            "I can't provide that response because it did not "
                            "pass the safety check."
                        )
                    )
                ]
        return None

## Chunk 5 — Create the agent

The `middleware` list attaches the guardrail to the agent. The judge model is separate from the main agent model.

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=tools,
    middleware=[SafetyGuardrailMiddleware()],
    system_prompt=(
        "You are a helpful demo assistant. Use the dummy tools when relevant. "
        "Never claim that a real email was sent; the email tool only drafts."
    ),
)

print("Agent created.")

## Chunk 6 — Test a normal request

This should usually produce a normal answer. The safety judge runs after the agent finishes.

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the demo weather in Bengaluru?"}
    ]
})

print("\nFinal assistant message:")
print(result["messages"][-1].content)

## Chunk 7 — Test the dummy email tool

This creates a **simulated draft only**. It does not send an email.

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Use the draft_email tool to draft a meeting update to "
                "demo@example.com with subject 'Meeting update' and body "
                "'The meeting starts at 3 PM.'"
            )
        }
    ]
})

print("\nFinal assistant message:")
print(result["messages"][-1].content)

## Chunk 8 — Understand the execution flow

```text
User request
    ↓
Agent model ↔ dummy tools (if needed)
    ↓
Agent reaches its final answer
    ↓
after_agent() runs
    ↓
Safety model judges the answer: SAFE / UNSAFE
    ├── SAFE   → return None; keep the answer
    └── UNSAFE → append a safe replacement message
    ↓
Return final state
```

### Key concepts

- `AgentMiddleware`: base class for custom agent middleware.
- `after_agent`: runs after the agent has completed its normal work.
- `AgentState`: provides access to the agent's state, including messages.
- `Runtime`: runtime context passed to the middleware hook.
- `@hook_config(can_jump_to=["end"])`: declares that the hook may jump to the graph's end.
- `return None`: means the middleware makes no state update.
- `return {"messages": [...]}`: updates the message state. Since messages are append-reduced, this example appends a replacement rather than deleting the earlier message from the internal history.

**Production note:** if you must guarantee that unsafe content is never exposed, validate the exact user-facing output at the serving boundary too. An LLM judge is probabilistic and can make mistakes.

## Optional exercise

1. Change the judge prompt to also flag disclosure of secrets.
2. Add a deterministic `before_agent` guardrail for a simple blocked keyword.
3. Compare a deterministic rule with an LLM-based judge.
4. Inspect the full `result["messages"]` list to see the conversation state.

For production, prefer deterministic checks for rules that can be stated precisely, and use an LLM judge only where semantic interpretation is necessary.